# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
PATH = '/content/drive/MyDrive/DS 2002/data/product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [3]:
poptarts = cat[cat["product_name"].str.contains("pop", case=False, na=False)]

print(f"Count: {len(poptarts)}")
poptarts

Count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


For this question, first a data frame is created that filters the `cat` data frame, based on `product_name`, seeing when the product's name contains `"pop"`. Any product names including "pop" will be included in the new data frame, so all the Poptart products will appear. Then, I printed both the lenth of the dataframe (how many Poptart products there are) and the data frame.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [4]:
strawberry = poptarts[poptarts["product_name"].str.contains("strawberry", case=False, na=False)]
not_strawberry = poptarts.drop(strawberry.index)

print("Strawberry Pop-Tart:")
print(strawberry[["sku", "product_name"]].to_string(index=False))

print(f"\nNot strawberry (kept separate): {len(not_strawberry)}")
print(not_strawberry[["sku", "product_name"]])

Strawberry Pop-Tart:
          sku         product_name
        PT-12 pop-tarts strawberry
   SKU#459812  Strawberry Pop-Tart
POPTART-STRAW Pop-Tarts Strawberry
PT_STRAWBERRY pop-tarts strawberry

Not strawberry (kept separate): 0
Empty DataFrame
Columns: [sku, product_name]
Index: []


**Pop-Tart rows that are a different product:** There are no poptart rows that are a different product, all of them are strawberry poptarts.

Here, I first filter `poptarts` to make the data frame `strawberry` which only keeps the rows that have product names that include the word strawberry, being case-insensitive and safe against missing values. Then, the `not_strawberry` data frame is made up of any Poptart rows that are not strawberry. Then `poptarts.drop(strawberry.index)` removes every row of `poptarts` whose index label also appears in `strawberry`. This guarantees the two groups don't overlap.

Then the print displays each group, which finds that all the rows in `poptarts` are strawberry.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [5]:
CANON = "PT-12"

sku_map = {
    "SKU#459812":    CANON,
    "POPTART-STRAW": CANON,
    "PT_STRAWBERRY": CANON,
}

cat["canonical_sku"] = cat["sku"].replace(sku_map)

print(cat.loc[cat["sku"].isin(sku_map) | (cat["sku"] == CANON), ["sku", "canonical_sku"]])

             sku canonical_sku
0          PT-12         PT-12
1     SKU#459812         PT-12
2  POPTART-STRAW         PT-12
3  PT_STRAWBERRY         PT-12


First, `CANON` holds the one SKU (`"PT-12"`, chosen because it's the cleanest of the four) that every variant SKU will collapse onto. Then, the `sku_map` dict directs the other 3 SKU's to the canonical value. Then, `cat["sku"].replace(sku_map)` applies the mapping onto `cat`, making a new `canonical_sku` column for the canonical SKUs. This new column has the SKUs from the dict, and any SKU that was not a key in the dict is left unchanged, rather than turning into NaN's. Now, no information is lost. There is no information deleted from the catalog and the original strawberry Poptart SKU values are preserved.


Then, to see the changes made, I print the rows of `cat` that were included in the `sku_map` dict, showing the `sku` and `canonical_sku` columns to show the original SKU's and the standardized canonical SKU's.



### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [6]:
is_pt = cat["canonical_sku"] == CANON

before = cat.loc[is_pt, "category"].value_counts(dropna=False)

cat["category_clean"] = cat["category"]        # everything else keeps its original label
cat.loc[is_pt, "category_clean"] = cat.loc[is_pt, "category"].str.strip().str.title()

after = cat.loc[is_pt, "category_clean"].value_counts(dropna=False)

print("BEFORE (strawberry Pop-Tart rows):\n", before.to_string())
print("\nAFTER (strawberry Pop-Tart rows):\n", after.to_string())

BEFORE (strawberry Pop-Tart rows):
 category
Breakfast          2
Breakfast Foods    1
breakfast          1

AFTER (strawberry Pop-Tart rows):
 category_clean
Breakfast          3
Breakfast Foods    1


First, `is_pt` is built off of any row in `cat` where the `canonical_sku` column equals `CANON`, resulting in the four strawberry Poptart rows. Then, to find the counts of each category before any change is made, I used `cat.loc[is_pt, "category"].value_counts(dropna=False)`, which selects the `category` column from the Poptart rows, and then counts the unique labels. This finds `"Breakfast"` twice, `"Breakfast Foods"` once, and `"breakfast"` once. `dropna=False` would have had any missing category showing up as its own line, but there were no missing values so this didn't occur.

Then, `cat["category_clean"] = cat["category"]` creates a new column that includes the cleaned names of the categories, starting as a copy of the original. Then, `cat.loc[is_pt, "category_clean"] = ...` overwrites the Poptart rows with the cleaned version of the category name, using `str.strip()` to remove any excess white space, and `.str.title()` standardizes capitalization.  

`after` counts the unique category names, but now on the cleaned columns. `"Breakfast"` and `"breakfast"` have merged, but `"Breakfast Foods"` has stayed the same. Strip and title-case only fix whitespaces and capitalization, so these two categories can not be merged from this alone, extra code would be required.

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [7]:
import re

PACK_WORDS = {"pk", "pack", "ct"}

def name_key(s):
    s = re.sub(r"[^a-z\s]", " ", s.lower())
    words = [w[:-1] if w.endswith("s") and len(w) > 3 else w
             for w in s.split()]
    return " ".join(sorted(w for w in words if w not in PACK_WORDS))

cat["normalized"] = cat["product_name"].map(name_key)

candidates = (cat.groupby("normalized")["sku"]
              .agg(list)
              .loc[lambda s: s.str.len() > 1])
candidates

,sku
normalized,
aa batterie,"[BATT-100, BAT-AA4, BATT_AA_4PK]"
aid first kit,"[FAID-01, FIRSTAID-KIT]"
beer light,"[BEER-24, LTBEER24]"
board game monopoly,"[BG-MONO, MONOPOLY-01, GAME-MONO]"
bottled water,"[WTR-24, WATER24PK]"
bread white,"[BREAD-WH, WHITEBREAD]"
butter peanut,"[PB-16, PNUTBTR-16]"
candle emergency,"[CNDL-EM, CANDLE-EMRG]"
canned soup,"[SOUP-CAN, CNSOUP-01]"


First, the `name_key` function turns each the product name's into a key. It starts with `re.sub(r"[^a-z\s]", " ", s.lower())`, which turns the name into lowercase and replaces characters that aren't a letter or a white space with a space. Punctuation and digits also become spaces.

The list also strips trailing s's from words, so singular and plural versions of the same product produce the same key. It only does it to words longer than 3 letters, so any short words that end in s that aren't plural are not impacted.

Then, any words in `PACK_WORDS` (pk, pack, ct) are removed, and everything remaining is sorted. This make it so the key is independent of word order and how the pack size is spelled.

`cat["name"].map(name_key)`applies the function to every name, and stores the result in `normalized` column. Then, the `groupby` collects the SKUs that share a key with `.agg(list)`. `loc[lambda s: s.str.len() > 1]` keeps only the keys with more than one SKU, which is what we're looking for.


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [8]:
assert cat["canonical_sku"].notna().all()
assert len(cat) == len(pd.read_csv(PATH))

### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

A: THe two product families that are clearly fragmenteed are Monopoly and AA batteries, both of which have 3 SKUs. I would do the same normalized-name grouping I used in the last step (lowercase, punctuation to spaces, drop pack words, sort the words). This puts all of the family's names under one key. I would also want to review the groups by hand after to make sure everything looks okay.

B: Consolidation becomes a business decision when the data can't tell you whether two rows are the same sellable unit. For example, in this data there is a 6 pack strawberry Poptart product (`POPTART-STRAW`), whereas the other three strawberry Poptart products are all 4 packs. I chose to map this product onto the same `PT-12` SKU as the other poptart products because the names match, but the different size packs could have important differences such as cost and inventory counts. By merging them together, the differences are hidden. I would ask whoever owns the master product data, such as a merchandising manager, whether a 6 pack and a 4 pack should count as separate SKUs. If they say yes,the 6 pack should separate from the 4 packs and have a separate canonical SKU.